# pm_helper: PowerModels.jl power flow solver
* conda: h-py312-basic
* notes and design details: [`work-notes/pm_helper.md`](../work-notes/pm_helper.md) (TODO)

Simple sanity-check notebook for the PowerModels.jl driver (`pm_solve.jl`), mirroring
the style of [`gridkit_helper.ipynb`](gridkit_helper.ipynb): one base case, one solve,
inspect the result. `pm_solve.jl` is a thin Julia wrapper around PowerModels.jl —
called via `subprocess`, same pattern as `solve_pf` in [`pf_helper.ipynb`](pf_helper.ipynb).

## workflow
1. **Section 1**: imports and path setup
2. **Section 2**: parse the base case and inspect the network dict
3. **Section 3**: solve AC power flow, display results
4. **Section 4**: solve DC power flow, compare vs AC
5. **Section 5**: base case, three-way comparison (PM.jl AC, GridKit `solve_pf`, TAMU/PowerWorld reference)
6. **Section 6**: perturbed-case sweep, PM.jl vs GridKit `solve_pf` on all 14 existing cases
7. **Section 7**: data-driven verdict - is GridKit's `solve_pf` reasonable for the Aleatoric UQ track?


In [ ]:
import os
import sys
import json
import importlib
from pathlib import Path

import pandas as pd
from IPython.core.interactiveshell import InteractiveShell

InteractiveShell.ast_node_interactivity = "all"
pd.set_option("display.max_rows", 10)
pd.set_option("display.max_columns", 20)
pd.set_option("display.float_format", "{:.6f}".format)

# === GridKit / pm-solver paths ===
GRIDKIT_REPO = Path.home() / "gridkit"
UQ_DIR = GRIDKIT_REPO / "uq-usecase"
PM_PROJECT_DIR = UQ_DIR / "pm-solver"
PM_SOLVE_JL = PM_PROJECT_DIR / "pm_solve.jl"
JULIA_BIN = Path.home() / "bin/julia112"

# === case data: reuse the .m files already generated for pf_helper.ipynb ===
PF_M_CASES = UQ_DIR / "pf-solver/m-cases"
BASECASE_M = PF_M_CASES / "basecase/case_ACTIVSg200.m"

# === py-utils on path ===
PY_UTILS_DIR = str(UQ_DIR / "py-utils")
if PY_UTILS_DIR not in sys.path:
    sys.path.insert(0, PY_UTILS_DIR)

import pf_utils, pm_utils

importlib.reload(pf_utils)
importlib.reload(pm_utils)
from pm_utils import run_pm_solve, run_pm_solve_out, pm_summary

for label, p in [
    ("julia112 binary", JULIA_BIN),
    ("pm-solver Project.toml", PM_PROJECT_DIR / "Project.toml"),
    ("pm_solve.jl", PM_SOLVE_JL),
    ("base case .m", BASECASE_M),
    ("pm_utils.py", UQ_DIR / "py-utils/pm_utils.py"),
]:
    status = "OK " if p.exists() else "MISSING"
    print(f"  [{status}]  {label}: {p}")

# Section 2: parse the base case and inspect the network dict

`pm_solve.jl` parses the `.m` file internally via `PowerModels.parse_file()` and prints
a one-line summary to stderr (bus/gen counts, offline gens, baseMVA). Run it once here
just to confirm the parse succeeds and see that summary, before solving anything.


In [ ]:
import subprocess

# Small inline Julia snippet: parse the case and print bus/gen/branch/load counts
# as JSON, so we can inspect the network dict without a full solve.
_inspect_jl = f"""
using PowerModels, JSON
PowerModels.silence()
data = PowerModels.parse_file("{BASECASE_M}")
summary = Dict(
    "n_bus" => length(data["bus"]),
    "n_gen" => length(data["gen"]),
    "n_branch" => length(data["branch"]),
    "n_load" => length(data["load"]),
    "n_gen_offline" => count(g -> g["gen_status"] == 0, values(data["gen"])),
    "baseMVA" => data["baseMVA"],
)
println(JSON.json(summary))
"""

r = subprocess.run(
    [str(JULIA_BIN), f"--project={PM_PROJECT_DIR}", "-e", _inspect_jl],
    capture_output=True,
    text=True,
)
print(r.stderr[-500:] if r.returncode != 0 else "")
network_summary = json.loads(r.stdout.strip().splitlines()[-1])
network_summary

# Section 3: solve AC power flow

## How the Python notebook calls Julia

Each call to `run_pm_solve` / `run_pm_solve_out` in `pm_utils.py` launches a fresh
`subprocess`:

```
julia112 --project=<pm-solver/> pm_solve.jl <input.m> [--output-m ...] [--tol ...] [--max-iter ...] [--flat-start]
```

- **`--project=pm-solver/`**: tells Julia to load exactly the packages pinned in
  `pm-solver/Project.toml` (PowerModels v0.21.6, Ipopt v1.15.0, JuMP v1.31.1).
  This is the Julia equivalent of a conda environment.
- **Precompilation**: Julia compiles each package to native code on first load and
  stores the result in `~/.julia/compiled/`. Subsequent calls reuse this cache, but
  the cache is reloaded from disk on every subprocess invocation.
- **Per-call overhead (measured on this node)**: ~8.5 s total wall time, of which
  ~5.5 s is Julia startup + loading the precompiled packages, and ~2.9 s is Ipopt
  solving the 200-bus AC PF. For the 14-case sweep this is ~2 min total. For 8760
  scenarios this cost would be prohibitive; the plan is a batched manifest mode
  where one Julia process loads the packages once and loops over all cases (see
  `plan.md` Phase D).
- **stdout / stderr split**: `pm_solve.jl` prints one `bus <i> V=... theta_deg=...
  type=...` line per bus to stdout; all diagnostics (solver params, Ipopt summary,
  termination status) go to stderr. `subprocess.run(capture_output=True)` keeps them
  separate.

## Warm start vs. cold start (`--flat-start`)

By default this is a **warm start**: `PowerModels.parse_file` reads the Vm/Va values
already embedded in the `.m` file into the data dict, and Ipopt uses those as its
initial point. For the raw base case (`case_ACTIVSg200.m`) those are the
TAMU/PowerWorld solved values (Vm≈1.01–1.05 pu, Va≈-3 to -11°), so Ipopt
converges in only 4 iterations.

For the perturbed cases the same TAMU Vm/Va are the initial point (the perturbation
scripts only modify PD/QD/PG, not bus Vm/Va), so it's a warm start from the
unperturbed solution — still close for mild perturbations, potentially farther for
stress cases like `load80pct`.

**`--flat-start`** (available via `flat_start=True` in `run_pm_solve`) resets all
buses to Vm=1.0, Va=0.0 before solving. This is useful to:
1. Check robustness: can PM.jl find the solution without a near-initial-point?
2. Match GridKit's `--flat-start` behavior for a fair head-to-head comparison.
3. Simulate the 8760 production scenario where PCM hourly `.m` files may not have
   reliable embedded Vm/Va values.


In [ ]:
# Solver parameters: adjust tol/max_iter here to experiment before running the sweep.
IPOPT_TOL = 1e-8  # Ipopt default; tighten to 1e-10 if solutions look borderline
IPOPT_MAX_ITER = 300

# Persist the solved .m alongside the base case, mirroring pf-solver/m-cases/'s
# <name>_solved.m convention (PM.jl's own m-cases tree).
PM_M_CASES = PM_PROJECT_DIR / "m-cases"
PM_BASECASE_SOLVED_M = PM_M_CASES / "basecase/case_ACTIVSg200_solved.m"
PM_BASECASE_SOLVED_M.parent.mkdir(parents=True, exist_ok=True)

ac_df, ac_stderr, ac_rc = run_pm_solve_out(
    JULIA_BIN,
    PM_PROJECT_DIR,
    PM_SOLVE_JL,
    BASECASE_M,
    PM_BASECASE_SOLVED_M,
    pf_type="ac",
    tol=IPOPT_TOL,
    max_iter=IPOPT_MAX_ITER,
)
pm_summary("ACTIVSg200 base case (PM.jl AC PF)", ac_df, ac_rc, ac_stderr)

# --- sanity checks vs TAMU/PowerWorld reference embedded in the raw .m ---
# The raw case file ships with Vm/Va already solved by the data provider (TAMU/PowerWorld);
# these are the closest thing to a "ground truth" for this synthetic network.
if ac_rc == 0:
    from pf_utils import parse_raw_m_bus, diff_vs_base

    tamu_df = parse_raw_m_bus(BASECASE_M)
    print("Sanity check: PM.jl vs TAMU/PowerWorld reference (embedded in raw .m):")
    _ = diff_vs_base("PM.jl vs TAMU", ac_df, tamu_df)

ac_df.head()

# Section 4: solve DC power flow, compare vs AC

DC power flow is a linearized, real-power-only approximation: voltage magnitude is
fixed at 1.0 p.u. everywhere and only angle is solved. It's much faster (no reactive
power, no Newton iterations on a nonlinear model) and is the same approximation
discussed in [`pf_helper.md` Section 8](../work-notes/pf_helper.md) as being
consistent with GridKit's angle-dominated PF response on this network.


In [ ]:
dc_df, dc_stderr, dc_rc = run_pm_solve(
    JULIA_BIN, PM_PROJECT_DIR, PM_SOLVE_JL, BASECASE_M, pf_type="dc"
)
pm_summary("ACTIVSg200 base case (DC PF)", dc_df, dc_rc, dc_stderr)
dc_df.head()

In [ ]:
cmp = ac_df.merge(
    dc_df[["bus_i", "theta_deg"]].rename(columns={"theta_deg": "theta_deg_dc"}),
    on="bus_i",
)
cmp["dTheta_ac_vs_dc"] = (cmp.theta_deg - cmp.theta_deg_dc).abs()

print("AC vs DC angle comparison (base case):")
print(f"  max |theta_AC - theta_DC| = {cmp.dTheta_ac_vs_dc.max():.4f} deg")
print(f"  mean |theta_AC - theta_DC| = {cmp.dTheta_ac_vs_dc.mean():.4f} deg")
print(f"  AC voltage range: [{ac_df.V_pu.min():.4f}, {ac_df.V_pu.max():.4f}] pu")
print("  DC voltage: fixed at 1.0 pu everywhere (not modeled)")
cmp.sort_values("dTheta_ac_vs_dc", ascending=False).head(10)

# Section 5: base case, three-way comparison (PM.jl, GridKit, TAMU/PowerWorld)

Cross-validates PM.jl's AC PF solution against two independent references:
- **GridKit `solve_pf`**: solved in `pf_helper.ipynb` Section 4, persisted to
  `pf-solver/m-cases/basecase/case_ACTIVSg200_solved.m`
  (V range [1.0091, 1.0432] pu, 0 violations, nni=4, ||f||=4.42e-6).
- **TAMU/PowerWorld reference**: Vm/Va embedded directly in the raw
  `case_ACTIVSg200.m` from the data provider (see [`pf_helper.md`](../work-notes/pf_helper.md)
  Section 4). Read with no solve via `parse_raw_m_bus`.


In [ ]:
import sys

if str(UQ_DIR / "py-utils") not in sys.path:
    sys.path.insert(0, str(UQ_DIR / "py-utils"))
from pf_utils import pf_summary, diff_vs_base, parse_raw_m_bus

# GridKit's solved base case was produced in pf_helper.ipynb (see Section 4 there)
# and persisted to pf-solver/m-cases/basecase/case_ACTIVSg200_solved.m.
# Read it directly via parse_raw_m_bus - no need to re-run solve_pf here.
GRIDKIT_SOLVED_M = PF_M_CASES / "basecase/case_ACTIVSg200_solved.m"
print(
    f"  [{'OK ' if GRIDKIT_SOLVED_M.exists() else 'MISSING'}]  GridKit solved base case: {GRIDKIT_SOLVED_M}"
)

gk_df = parse_raw_m_bus(GRIDKIT_SOLVED_M)
print(
    f"  GridKit solve_pf base case: V range [{gk_df.V_pu.min():.4f}, {gk_df.V_pu.max():.4f}] pu"
)

# TAMU/PowerWorld reference: embedded in the raw unsolved case file
tamu_df = parse_raw_m_bus(BASECASE_M)
print(
    f"  TAMU/PowerWorld reference:  V range [{tamu_df.V_pu.min():.4f}, {tamu_df.V_pu.max():.4f}] pu"
)

In [ ]:
print(
    "=== Base case: three-way comparison (PM.jl AC, GridKit solve_pf, TAMU/PowerWorld) ===\n"
)
pm_vs_gk = diff_vs_base("PM.jl (AC) vs GridKit solve_pf", ac_df, gk_df)
pm_vs_tamu = diff_vs_base("PM.jl (AC) vs TAMU/PowerWorld reference", ac_df, tamu_df)
gk_vs_tamu = diff_vs_base(
    "GridKit solve_pf vs TAMU/PowerWorld reference", gk_df, tamu_df
)

pm_vs_gk.sort_values("dV", ascending=False).head(10)

## Section 5 findings: GridKit voltage bias confirmed by PM.jl

The three-way comparison gives a clean result. All numbers are from the cell above.

| pair | max \|dV\| (pu) | mean \|dV\| (pu) | max \|dTheta\| (deg) | mean \|dTheta\| (deg) |
|---|---|---|---|---|
| PM.jl vs TAMU/PowerWorld | 0.000011 | 0.000002 | 0.000926 | 0.000460 |
| PM.jl vs GridKit `solve_pf` | 0.029728 | 0.005057 | 0.190703 | 0.073286 |
| GridKit vs TAMU/PowerWorld | 0.029730 | 0.005056 | 0.190720 | 0.073687 |

**PM.jl reproduces the TAMU/PowerWorld solution to numerical noise** (max |dV| = 1.1e-5 pu,
max |dTheta| = 9.3e-4 deg). This confirms PM.jl is correct and that the TAMU/PowerWorld
embedded Vm/Va are the same operating point.

**GridKit deviates from both PM.jl and PowerWorld by the same amount** (~0.030 pu,
~0.19 deg). The deviation is entirely in GridKit, not between the two independent solvers.
Both solvers warm-start from the same TAMU/PowerWorld Vm/Va in the `.m` file, so this
is a genuine model difference, not an initialization artifact. The cause is GridKit's
non-enforcement of generator reactive power limits (`Qmax`/`Qmin`), see
[`pf_helper.md` Section 6](../work-notes/pf_helper.md) for details.

**Split verdict for the UQ track:**
- **Angles**: max |dTheta| = 0.19 deg, well within the 1.0 deg threshold. GridKit
  is suitable for angle-based QoIs (line flows, angle differences, rotor angle
  initial conditions).
- **Voltages**: max |dV| = 0.030 pu, 3x above the 0.01 pu threshold. GridKit is
  **not** suitable for voltage-magnitude QoIs (voltage constraint checking, reactive
  dispatch, voltage stability margin, Vr/Vi patching into `illinois.json`). PM.jl
  should be used as the primary solver if accurate voltages are needed.

This conclusion is data-driven and depends on the base-case result above. Section 6
checks whether it holds across all 14 perturbed cases.



# Section 6: perturbed-case sweep (PM.jl vs GridKit `solve_pf`)

Repeat the comparison across the 14 already-generated perturbed cases in
[`pf-solver/m-cases`](../pf-solver/m-cases) (5 load levels, 4 wind-curtailment
levels, 5 generator-outage scenarios). The raw `.m` case files and GridKit's
`*_solved.m` files were produced in
[`pf_helper.ipynb`](../notebooks/pf_helper.ipynb) Sections 10-12. GridKit's
already-solved `*_solved.m` files are reused directly (read via `parse_raw_m_bus`,
no re-solve) rather than re-running `solve_pf`; only PM.jl needs to solve each
raw case fresh.


In [ ]:
PERTURBED_CASES = [
    "gen2rand_off",
    "gen3rand_off",
    "gen5rand_off",
    "gen10rand_off",
    "gen147off",
    "load5pct",
    "load10pct",
    "load20pct",
    "load40pct",
    "load80pct",
    "wind10pct",
    "wind20pct",
    "wind40pct",
    "wind80pct",
]

sweep_rows = []
for case in PERTURBED_CASES:
    raw_m = PF_M_CASES / f"case_ACTIVSg200_{case}.m"
    gk_solved_m = PF_M_CASES / f"case_ACTIVSg200_{case}_solved.m"
    if not raw_m.exists() or not gk_solved_m.exists():
        print(f"  [SKIP] {case}: missing raw or GridKit-solved .m")
        continue

    pm_df, pm_stderr, pm_rc = run_pm_solve(
        JULIA_BIN,
        PM_PROJECT_DIR,
        PM_SOLVE_JL,
        raw_m,
        pf_type="ac",
        tol=IPOPT_TOL,
        max_iter=IPOPT_MAX_ITER,
    )
    gk_case_df = parse_raw_m_bus(gk_solved_m)

    row = {"case": case, "pm_converged": pm_rc == 0}
    if pm_rc == 0:
        import re as _re

        iters_m = _re.search(r"Number of Iterations\.+:\s*(\d+)", pm_stderr)
        row["ipopt_iters"] = int(iters_m.group(1)) if iters_m else None
        cmp = pm_df.merge(
            gk_case_df[["bus_i", "V_pu", "theta_deg"]].rename(
                columns={"V_pu": "V_gk", "theta_deg": "theta_gk"}
            ),
            on="bus_i",
        )
        cmp["dV"] = (cmp.V_pu - cmp.V_gk).abs()
        cmp["dTheta"] = (cmp.theta_deg - cmp.theta_gk).abs()
        row["max_dV_pu"] = cmp.dV.max()
        row["mean_dV_pu"] = cmp.dV.mean()
        row["max_dTheta_deg"] = cmp.dTheta.max()
        row["mean_dTheta_deg"] = cmp.dTheta.mean()
    sweep_rows.append(row)

sweep_df = pd.DataFrame(sweep_rows)
sweep_df

## Section 6 findings: voltage bias is a stable, case-independent offset

All 14 perturbed cases converged (4 Ipopt iterations each, same as the base case).
All numbers in this cell come directly from `sweep_df` above.

**Voltage bias is essentially constant across all operating conditions.**
`max_dV_pu` (PM.jl vs GridKit) ranges from 0.027 to 0.030 pu across all 14 cases,
indistinguishable from the base-case value of 0.030 pu. Even the most aggressive
perturbations (load80pct: +80% load, gen10rand_off: 10 generators out) do not grow
the voltage discrepancy. This confirms the bias is a systematic, case-independent
offset arising from GridKit's model (unconditional PV enforcement, no Q-limit
switching), not a sensitivity to operating point.

**Angle differences grow with perturbation stress but stay within threshold.**
`max_dTheta_deg` ranges from 0.19 deg (mild perturbations) to 0.23 deg (gen10rand_off,
wind80pct), all well within the 1.0 deg acceptance threshold.

**Consequence for relative comparisons (base vs perturbed).**
Because the voltage bias is constant, it cancels when taking `diff_vs_base`: GridKit's
`ΔV = V_perturbed - V_base` is reliable even though the absolute voltages are offset.
GridKit is suitable for UQ sensitivity studies that measure *changes* in voltage rather
than absolute values. For absolute voltage accuracy (constraint checking, JSON patching),
use PM.jl.

**PM.jl independently confirms the voltage-stiff (DC-approximation-valid, angle-dominated)
regime found in [`pf_helper.ipynb`](pf_helper.ipynb) Sections 10-13.**
Because the (PM.jl vs GridKit) offset is constant, PM.jl's own
`ΔV = V_perturbed - V_base ≈ GridKit's ΔV`, the offset cancels in the delta. This means
the pattern found in GridKit (`max|dV| < 0.010 pu` even at ±80% load or 306 MW generation
loss) is not a GridKit modeling artifact. A full MATPOWER-equivalent solver with Q-limit
enforcement, given the same cases, reaches the same conclusion: on ACTIVSg200 with 24.5%
regulated buses, real-power perturbations manifest primarily as angle shifts. PM.jl
converging in 4 iterations for every one of the 14 cases (identical to the base case)
is further evidence that all operating points are in the same linear, voltage-stiff
regime. See [`pf_helper.md` Section 8](../work-notes/pf_helper.md) for the physical
explanation and Section 6 for the full cross-validation summary.


# Section 7: is GridKit's `solve_pf` reasonable for the Aleatoric UQ track?

**Context from Sections 5-6**: PM.jl and GridKit differ by a constant ~0.030 pu voltage
offset present at the base case and unchanged across all 14 perturbed cases. This offset
is a GridKit model artifact (working hypothesis: no Q-limit enforcement), not a
perturbation-induced divergence. PM.jl reproduces the TAMU/PowerWorld reference to
numerical noise (1.1e-5 pu), so the offset is entirely on the GridKit side.

Given this, the right question is not "do PM.jl and GridKit agree within a single
threshold?" but a **split verdict** across three separate questions:

1. **Angle accuracy**: does GridKit reproduce PM.jl bus angles within 1.0 deg across
   all 14 cases? Angle errors are small in Section 5 (0.19 deg base case) and grow
   modestly with stress. This determines whether GridKit is usable for angle-based QoIs.

2. **Absolute voltage accuracy**: does GridKit's |V| match PM.jl within 0.01 pu? This
   is expected to fail for all 14 cases due to the constant baseline offset — that is
   already a known result from Section 5, not new information.

3. **Relative ΔV accuracy**: is the constant offset truly constant, so it cancels in
   `ΔV = V_perturbed - V_base`? Section 6 confirmed yes (offset range 0.027-0.030 pu,
   variation < 0.003 pu across all cases). This determines whether GridKit is usable
   for voltage-sensitivity UQ.

The code below computes all three verdicts from `sweep_df` and `pm_vs_gk`.


In [ ]:
DTHETA_THRESH_DEG = 1.0  # angle agreement threshold
DV_ABS_THRESH_PU = 0.01  # absolute voltage agreement threshold
DV_OFFSET_VAR_PU = 0.005  # max allowed variation in (PM - GK) offset across cases
# (measures whether offset is truly constant / cancels in delta)
STRESS_CASES = ["load80pct", "wind80pct", "gen147off", "gen10rand_off"]

n_total = len(sweep_df)
conv_df = sweep_df[sweep_df.pm_converged].copy()
n_converged = len(conv_df)
print(f"PM.jl converged on {n_converged}/{n_total} perturbed cases.")
if n_converged < n_total:
    failed = sweep_df.loc[~sweep_df.pm_converged, "case"].tolist()
    print(f"  Non-converged: {failed}")

print()

# --- Verdict 1: Angle accuracy ---
angle_ok = conv_df.max_dTheta_deg < DTHETA_THRESH_DEG
n_angle_ok = angle_ok.sum()
print(
    f"[ANGLE ACCURACY]  max|dTheta| < {DTHETA_THRESH_DEG} deg: {n_angle_ok}/{n_converged} cases pass"
)
print(
    f"  Overall max|dTheta|: {conv_df.max_dTheta_deg.max():.4f} deg  (base case: {pm_vs_gk.dTheta.max():.4f} deg)"
)
if n_angle_ok == n_converged:
    print(
        "  PASS: GridKit angles agree with PM.jl (MATPOWER reference) within threshold."
    )
    print(
        "        GridKit is suitable for angle-based QoIs (line flows, rotor angle ICs)."
    )
else:
    failed_angle = conv_df.loc[~angle_ok, "case"].tolist()
    print(f"  FAIL: {len(failed_angle)} case(s) exceed angle threshold: {failed_angle}")

print()

# --- Verdict 2: Absolute voltage accuracy ---
volt_ok = conv_df.max_dV_pu < DV_ABS_THRESH_PU
n_volt_ok = volt_ok.sum()
base_volt_ok = pm_vs_gk.dV.max() < DV_ABS_THRESH_PU
print(
    f"[ABSOLUTE VOLTAGE] max|dV| < {DV_ABS_THRESH_PU} pu:  {n_volt_ok}/{n_converged} cases pass (base case: {'pass' if base_volt_ok else 'FAIL'})"
)
print(
    f"  Offset range across all 14 cases: [{conv_df.max_dV_pu.min():.4f}, {conv_df.max_dV_pu.max():.4f}] pu"
)
print(f"  Base case offset (Section 5):      {pm_vs_gk.dV.max():.6f} pu")
print(
    "  FAIL (expected): constant ~0.030 pu model offset (GridKit no Q-limit enforcement)."
)
print("        Use PM.jl for any QoI requiring accurate absolute voltage magnitudes.")
print(
    "        (Vr/Vi patching into illinois.json, constraint checking, voltage stability.)"
)

print()

# --- Verdict 3: Relative ΔV accuracy (does offset cancel in diffs?) ---
base_offset = pm_vs_gk.dV.max()
offset_variation = conv_df.max_dV_pu.max() - conv_df.max_dV_pu.min()
delta_v_ok = offset_variation < DV_OFFSET_VAR_PU
print(
    f"[RELATIVE dV]  offset variation across 14 cases: {offset_variation:.4f} pu  (threshold: < {DV_OFFSET_VAR_PU} pu)"
)
print(
    f"  Base-case offset: {base_offset:.4f} pu,  perturbed-case range: [{conv_df.max_dV_pu.min():.4f}, {conv_df.max_dV_pu.max():.4f}] pu"
)
if delta_v_ok:
    print(
        "  PASS: offset is effectively constant; cancels in ΔV = V_perturbed - V_base."
    )
    print(
        "        GridKit is suitable for voltage-sensitivity UQ (relative changes, not absolute values)."
    )
else:
    print(
        "  WARN: offset varies significantly across cases; ΔV may carry systematic error."
    )

print()

# --- Stress cases ---
stress_df = conv_df[conv_df.case.isin(STRESS_CASES)].copy()
stress_df["angle_ok"] = stress_df.max_dTheta_deg < DTHETA_THRESH_DEG
print("Stress cases:")
print(
    stress_df[["case", "max_dV_pu", "max_dTheta_deg", "angle_ok"]].to_string(
        index=False
    )
)

print()

# --- Final summary ---
print("=" * 70)
print("VERDICT SUMMARY")
print("=" * 70)
v1 = "PASS" if n_angle_ok == n_converged else "FAIL"
v2 = "FAIL (known model offset)"
v3 = "PASS" if delta_v_ok else "WARN"
print(f"  Angle accuracy (QoI: line flow, rotor angle IC):  {v1}")
print(f"  Absolute voltage accuracy (QoI: Vm constraint):  {v2}")
print(f"  Relative dV accuracy (QoI: voltage sensitivity): {v3}")
print()
if n_angle_ok == n_converged and delta_v_ok:
    print(
        "GridKit's solve_pf is suitable for the Aleatoric UQ track for angle-based and\n"
        "voltage-sensitivity QoIs. It must NOT be used for absolute voltage magnitudes\n"
        "(use PM.jl instead). This conclusion holds across the base case and all 14\n"
        "synthetic stress cases; re-check when real 8760-scenario PCM cases are available."
    )
else:
    print("See individual verdict lines above for details.")

# Section 8: high-perturbation stress tests (PM.jl only)

Goal: find where ACTIVSg200 hits its **convergence limit** and where voltages begin to
collapse, using PM.jl as the reference solver throughout. No GridKit comparison here.

Two sub-experiments:
- **8a: uniform load scaling** — multiply all bus Pd/Qd by a scalar (0.1x → 8x). Keeps
  load distribution shape fixed; slack absorbs the real power difference. Traces the PV
  curve toward the nose.
- **8b: largest-gen outage sweep** — take out the 1, 2, 5, 10, 15, 20, 25, 30 largest
  non-slack online generators by base-case Pg. Largest-first is deterministic and maximally
  stressful.

Both warm-start (base-case Vm/Va embedded in the `.m` file) **and** flat-start (Vm=1,
Va=0) are run for every case. Warm-start gives PM.jl the best chance of finding the
physical solution; flat-start checks whether a second solution branch exists (as both
GridKit and PM.jl showed in `pf_helper.ipynb` Section 14 and Section 8a below). If warm
and flat agree, the solution is unique in that region; if they disagree, two branches
exist (see [`pf_helper.md` Section 7](../work-notes/pf_helper.md)).

`.m` files are written to `pm-solver/m-cases/stress-test/` and kept for reruns.


In [17]:
import importlib
import m_viz_utils as _m_viz_utils
import pf_utils as _pf_utils

importlib.reload(_pf_utils)
importlib.reload(_m_viz_utils)

from pf_utils import make_load_scale_m, make_gen_off_largest_m
from m_viz_utils import read_matpower_case, plot_gen_dispatch

# Base case parsed for viz (genfuel, gen MW, highlight_buses support)
il_case = read_matpower_case(BASECASE_M)
base_total_pd = il_case.bus["PD"].sum()
print(f"Base case: {len(il_case.bus)} buses, {len(il_case.gen)} gens")
print(f"Base total Pd = {base_total_pd:.1f} MW")

# Output directory for stress-test .m files
ST_DIR = PM_M_CASES / "stress-test"
ST_DIR.mkdir(parents=True, exist_ok=True)
print(f"\nStress-test .m dir: {ST_DIR}")

# --- Generate load-scale .m files ---
LOAD_SCALES = [0.1, 0.5, 1.0, 1.5, 2.0, 3.0, 4.0, 6.0, 8.0]
scale_m = {}  # scale -> Path
scale_pd = {}  # scale -> total Pd MW
print("\nGenerating load-scale cases:")
for s in LOAD_SCALES:
    p = ST_DIR / f"case_ACTIVSg200_load{s}x.m"
    scale_pd[s] = make_load_scale_m(BASECASE_M, p, scale=s)
    scale_m[s] = p

# --- Generate gen-off-largest .m files ---
GEN_OFF_NS = [1, 2, 5, 10, 15, 20, 25, 30]
genoff_m = {}  # n -> Path
genoff_buses = {}  # n -> frozenset
genoff_mw = {}  # n -> float
print("\nGenerating largest-gen-off cases:")
for n in GEN_OFF_NS:
    p = ST_DIR / f"case_ACTIVSg200_gen{n}largest_off.m"
    buses, mw = make_gen_off_largest_m(BASECASE_M, p, n_largest=n)
    genoff_m[n] = p
    genoff_buses[n] = buses
    genoff_mw[n] = mw

<module 'pf_utils' from '/home/isatkaus/gridkit/uq-usecase/py-utils/pf_utils.py'>

<module 'm_viz_utils' from '/home/isatkaus/gridkit/uq-usecase/py-utils/m_viz_utils.py'>

Base case: 200 buses, 49 gens
Base total Pd = 1475.7 MW

Stress-test .m dir: /home/isatkaus/gridkit/uq-usecase/pm-solver/m-cases/stress-test

Generating load-scale cases:
Written: case_ACTIVSg200_load0.1x.m  (scale=0.1x, total Pd=147.6 MW)
Written: case_ACTIVSg200_load0.5x.m  (scale=0.5x, total Pd=737.8 MW)
Written: case_ACTIVSg200_load1.0x.m  (scale=1.0x, total Pd=1475.7 MW)
Written: case_ACTIVSg200_load1.5x.m  (scale=1.5x, total Pd=2213.5 MW)
Written: case_ACTIVSg200_load2.0x.m  (scale=2.0x, total Pd=2951.4 MW)
Written: case_ACTIVSg200_load3.0x.m  (scale=3.0x, total Pd=4427.1 MW)
Written: case_ACTIVSg200_load4.0x.m  (scale=4.0x, total Pd=5902.8 MW)
Written: case_ACTIVSg200_load6.0x.m  (scale=6.0x, total Pd=8854.1 MW)
Written: case_ACTIVSg200_load8.0x.m  (scale=8.0x, total Pd=11805.5 MW)

Generating largest-gen-off cases:
Written: case_ACTIVSg200_gen1largest_off.m  (1 largest gens offline: buses [105]  |  MW dropped: 154.8)
Written: case_ACTIVSg200_gen2largest_off.m  (2 largest gens o

## Section 8a: uniform load scaling

Scale factors and corresponding total system load:


In [18]:
from IPython.display import Markdown, display as _display

# Print scale table before running any solves
_lines = [
    "| scale | total Pd (MW) | note |",
    "|------:|-------------:|------|",
]
for s in LOAD_SCALES:
    note = "base case (sanity check)" if s == 1.0 else ""
    _lines.append(f"| {s}x | {scale_pd[s]:.1f} | {note} |")
_display(Markdown("\n".join(_lines)))

| scale | total Pd (MW) | note |
|------:|-------------:|------|
| 0.1x | 147.6 |  |
| 0.5x | 737.8 |  |
| 1.0x | 1475.7 | base case (sanity check) |
| 1.5x | 2213.5 |  |
| 2.0x | 2951.4 |  |
| 3.0x | 4427.1 |  |
| 4.0x | 5902.8 |  |
| 6.0x | 8854.1 |  |
| 8.0x | 11805.5 |  |

In [23]:
import re as _re


def _pm_metrics(df, stderr, rc, base_df=None):
    """Extract scalar metrics from a PM.jl solve result."""
    if rc != 0 or df is None or df.empty:
        return {"converged": False}
    iters_m = _re.search(r"Number of Iterations\.+:\s*(\d+)", stderr)
    m = {
        "converged": True,
        "ipopt_iters": int(iters_m.group(1)) if iters_m else None,
        "V_min": df.V_pu.min(),
        "V_max": df.V_pu.max(),
        "n_violations": int(((df.V_pu < 0.95) | (df.V_pu > 1.05)).sum()),
    }
    if base_df is not None:
        merged = df.merge(
            base_df[["bus_i", "V_pu", "theta_deg"]].rename(
                columns={"V_pu": "V_base", "theta_deg": "theta_base"}
            ),
            on="bus_i",
        )
        m["max_dV_vs_base"] = (merged.V_pu - merged.V_base).abs().max()
        m["max_dTheta_vs_base"] = (merged.theta_deg - merged.theta_base).abs().max()
    return m


load_scale_rows = []
for s in LOAD_SCALES:
    m_path = scale_m[s]
    # PM.jl-solved output files (distinct from GridKit's *_solved.m written by
    # pf_helper.ipynb Section 8; those live in the same ST_DIR).
    pm_warm_solved = ST_DIR / f"case_ACTIVSg200_load{s}x_pm_solved.m"
    pm_flat_solved = ST_DIR / f"case_ACTIVSg200_load{s}x_pm_flat_solved.m"

    print(f"  [{s}x]  {scale_pd[s]:.0f} MW  warm...", end=" ", flush=True)
    df_w, err_w, rc_w = run_pm_solve_out(
        JULIA_BIN,
        PM_PROJECT_DIR,
        PM_SOLVE_JL,
        m_path,
        pm_warm_solved,
        pf_type="ac",
        tol=IPOPT_TOL,
        max_iter=IPOPT_MAX_ITER,
    )
    print(f"{'OK' if rc_w==0 else 'FAIL'}  flat...", end=" ", flush=True)
    df_f, err_f, rc_f = run_pm_solve_out(
        JULIA_BIN,
        PM_PROJECT_DIR,
        PM_SOLVE_JL,
        m_path,
        pm_flat_solved,
        pf_type="ac",
        tol=IPOPT_TOL,
        max_iter=IPOPT_MAX_ITER,
        flat_start=True,
    )
    print(f"{'OK' if rc_f==0 else 'FAIL'}")

    w = _pm_metrics(df_w, err_w, rc_w, base_df=ac_df)
    f = _pm_metrics(df_f, err_f, rc_f, base_df=ac_df)

    # If both warm and flat converged, check if they agree with each other
    agree = None
    if w["converged"] and f["converged"]:
        m2 = df_w.merge(
            df_f[["bus_i", "V_pu", "theta_deg"]].rename(
                columns={"V_pu": "V_f", "theta_deg": "theta_f"}
            ),
            on="bus_i",
        )
        agree = float((m2.V_pu - m2.V_f).abs().max())

    load_scale_rows.append(
        {
            "scale": s,
            "total_Pd_MW": scale_pd[s],
            "warm_conv": w.get("converged"),
            "warm_iters": w.get("ipopt_iters"),
            "warm_V_min": w.get("V_min"),
            "warm_V_max": w.get("V_max"),
            "warm_viols": w.get("n_violations"),
            "warm_dV_vs_base": w.get("max_dV_vs_base"),
            "warm_dTheta_vs_base": w.get("max_dTheta_vs_base"),
            "flat_conv": f.get("converged"),
            "flat_iters": f.get("ipopt_iters"),
            "flat_V_min": f.get("V_min"),
            "flat_V_max": f.get("V_max"),
            "flat_viols": f.get("n_violations"),
            "max_dV_warm_vs_flat": agree,
        }
    )

load_scale_df = pd.DataFrame(load_scale_rows)
load_scale_df

  [0.1x]  148 MW  warm... OK  flat... OK
  [0.5x]  738 MW  warm... OK  flat... OK
  [1.0x]  1476 MW  warm... OK  flat... OK
  [1.5x]  2214 MW  warm... OK  flat... OK
  [2.0x]  2951 MW  warm... OK  flat... OK
  [3.0x]  4427 MW  warm... FAIL  flat... FAIL
  [4.0x]  5903 MW  warm... FAIL  flat... FAIL
  [6.0x]  8854 MW  warm... FAIL  flat... FAIL
  [8.0x]  11806 MW  warm... FAIL  flat... FAIL


,scale,total_Pd_MW,warm_conv,warm_iters,warm_V_min,warm_V_max,warm_viols,warm_dV_vs_base,warm_dTheta_vs_base,flat_conv,flat_iters,flat_V_min,flat_V_max,flat_viols,max_dV_warm_vs_flat
0,0.100000,147.569000,True,4.000000,1.035678,1.077872,85.000000,0.037585,28.139960,True,4.000000,0.994028,1.034064,0.000000,0.043808
1,0.500000,737.845000,True,4.000000,1.033409,1.072671,44.000000,0.023180,15.830547,True,4.000000,0.991841,1.029221,0.000000,0.043450
2,1.000000,1475.690000,True,4.000000,1.010229,1.055588,1.000000,0.000000,0.000000,True,4.000000,0.967465,1.011182,0.000000,0.044407
3,1.500000,2213.535000,True,4.000000,0.979793,1.041472,0.000000,0.034954,17.328486,True,4.000000,0.934642,1.000000,31.000000,0.047750
4,2.000000,2951.380000,True,5.000000,0.925815,1.041472,31.000000,0.096442,38.352335,True,5.000000,0.869788,1.000000,141.000000,0.059453
5,3.000000,4427.070000,False,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN
6,4.000000,5902.760000,False,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN
7,6.000000,8854.140000,False,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN
8,8.000000,11805.520000,False,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN


## Section 8b: largest-gen outage sweep

Generators are taken offline largest-first by base-case Pg. Slack bus is excluded.
The table below shows which buses go offline and total MW dropped for each N.


In [24]:
# Markdown table: which buses go offline and MW dropped for each N
_lines = [
    "| N off | Buses offline | MW dropped |",
    "|------:|---------------|------------|",
]
for n in GEN_OFF_NS:
    buses_str = "{" + ", ".join(str(b) for b in sorted(genoff_buses[n])) + "}"
    _lines.append(f"| {n} | {buses_str} | {genoff_mw[n]:.1f} MW |")
_display(Markdown("\n".join(_lines)))

| N off | Buses offline | MW dropped |
|------:|---------------|------------|
| 1 | {105} | 154.8 MW |
| 2 | {105, 135} | 288.7 MW |
| 5 | {105, 115, 135, 136, 147} | 648.5 MW |
| 10 | {65, 104, 105, 115, 125, 126, 127, 135, 136, 147} | 919.7 MW |
| 15 | {65, 68, 104, 105, 115, 125, 126, 127, 135, 136, 147, 152, 153, 154, 155} | 1020.8 MW |
| 20 | {65, 68, 69, 70, 71, 72, 73, 104, 105, 115, 125, 126, 127, 135, 136, 147, 152, 153, 154, 155} | 1062.7 MW |
| 25 | {65, 68, 69, 70, 71, 72, 73, 94, 104, 105, 115, 125, 126, 127, 135, 136, 147, 152, 153, 154, 155, 167, 170, 182, 183} | 1086.9 MW |
| 30 | {53, 65, 67, 68, 69, 70, 71, 72, 73, 91, 94, 104, 105, 114, 115, 125, 126, 127, 135, 136, 147, 151, 152, 153, 154, 155, 167, 170, 182, 183} | 1095.6 MW |

In [25]:
# Base-case dispatch overview with the 30 largest gens highlighted
# (shows which generators will progressively go offline across the sweep)
fig_genoff_overview = plot_gen_dispatch(
    il_case,
    title="ACTIVSg200 base-case dispatch — largest-gen outage sweep context\n"
    "(generators taken offline largest-first; top-30 by Pg highlighted)",
    highlight_buses=sorted(genoff_buses[30]),
)
_ = fig_genoff_overview.show()

In [26]:
gen_off_rows = []
for n in GEN_OFF_NS:
    m_path = genoff_m[n]
    pm_warm_solved = ST_DIR / f"case_ACTIVSg200_gen{n}largest_off_pm_solved.m"
    pm_flat_solved = ST_DIR / f"case_ACTIVSg200_gen{n}largest_off_pm_flat_solved.m"

    print(f"  [N={n:2d}]  {genoff_mw[n]:.0f} MW dropped  warm...", end=" ", flush=True)
    df_w, err_w, rc_w = run_pm_solve_out(
        JULIA_BIN,
        PM_PROJECT_DIR,
        PM_SOLVE_JL,
        m_path,
        pm_warm_solved,
        pf_type="ac",
        tol=IPOPT_TOL,
        max_iter=IPOPT_MAX_ITER,
    )
    print(f"{'OK' if rc_w==0 else 'FAIL'}  flat...", end=" ", flush=True)
    df_f, err_f, rc_f = run_pm_solve_out(
        JULIA_BIN,
        PM_PROJECT_DIR,
        PM_SOLVE_JL,
        m_path,
        pm_flat_solved,
        pf_type="ac",
        tol=IPOPT_TOL,
        max_iter=IPOPT_MAX_ITER,
        flat_start=True,
    )
    print(f"{'OK' if rc_f==0 else 'FAIL'}")

    w = _pm_metrics(df_w, err_w, rc_w, base_df=ac_df)
    f = _pm_metrics(df_f, err_f, rc_f, base_df=ac_df)

    agree = None
    if w["converged"] and f["converged"]:
        m2 = df_w.merge(
            df_f[["bus_i", "V_pu", "theta_deg"]].rename(
                columns={"V_pu": "V_f", "theta_deg": "theta_f"}
            ),
            on="bus_i",
        )
        agree = float((m2.V_pu - m2.V_f).abs().max())

    gen_off_rows.append(
        {
            "n_off": n,
            "MW_lost": genoff_mw[n],
            "buses_off": sorted(genoff_buses[n]),
            "warm_conv": w.get("converged"),
            "warm_iters": w.get("ipopt_iters"),
            "warm_V_min": w.get("V_min"),
            "warm_V_max": w.get("V_max"),
            "warm_viols": w.get("n_violations"),
            "warm_dV_vs_base": w.get("max_dV_vs_base"),
            "warm_dTheta_vs_base": w.get("max_dTheta_vs_base"),
            "flat_conv": f.get("converged"),
            "flat_iters": f.get("ipopt_iters"),
            "flat_V_min": f.get("V_min"),
            "flat_V_max": f.get("V_max"),
            "flat_viols": f.get("n_violations"),
            "max_dV_warm_vs_flat": agree,
        }
    )

gen_off_df = pd.DataFrame(gen_off_rows)
gen_off_df.drop(columns=["buses_off"])  # show numeric summary; buses_off is verbose

  [N= 1]  155 MW dropped  warm... 

OK  flat... OK
  [N= 2]  289 MW dropped  warm... OK  flat... OK
  [N= 5]  649 MW dropped  warm... OK  flat... OK
  [N=10]  920 MW dropped  warm... OK  flat... OK
  [N=15]  1021 MW dropped  warm... OK  flat... OK
  [N=20]  1063 MW dropped  warm... OK  flat... OK
  [N=25]  1087 MW dropped  warm... OK  flat... FAIL
  [N=30]  1096 MW dropped  warm... FAIL  flat... FAIL


,n_off,MW_lost,warm_conv,warm_iters,warm_V_min,warm_V_max,warm_viols,warm_dV_vs_base,warm_dTheta_vs_base,flat_conv,flat_iters,flat_V_min,flat_V_max,flat_viols,max_dV_warm_vs_flat
0,1,154.800000,True,4.000000,1.009945,1.055087,1.000000,0.004618,7.593108,True,4.000000,0.967127,1.010553,0.000000,0.044534
1,2,288.720000,True,4.000000,1.006312,1.053886,1.000000,0.010709,9.569686,True,4.000000,0.963055,1.009132,0.000000,0.044753
2,5,648.540000,True,4.000000,0.989179,1.045690,0.000000,0.038992,20.970883,True,4.000000,0.941857,1.000000,18.000000,0.047827
3,10,919.700000,True,5.000000,0.840948,1.041472,85.000000,0.177366,29.009222,True,5.000000,0.756803,1.000000,117.000000,0.084145
4,15,1020.760000,True,5.000000,0.820917,1.041472,96.000000,0.196799,31.310042,True,6.000000,0.723126,1.000000,152.000000,0.097791
5,20,1062.660000,True,5.000000,0.809003,1.041472,99.000000,0.208330,32.423371,True,6.000000,0.699006,1.000000,182.000000,0.109997
6,25,1086.930000,True,6.000000,0.736386,1.040559,177.000000,0.278661,36.277240,False,NaN,NaN,NaN,NaN,NaN
7,30,1095.580000,False,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN


## Section 8a findings: load scaling — PV-curve nose and regime boundaries

All numbers come directly from `load_scale_df` above.

| scale | total Pd (MW) | warm conv | warm V_min (pu) | warm violations | warm dV vs base (pu) | warm dTheta vs base (deg) | flat conv | warm vs flat (pu) |
|------:|-------------:|:---------:|----------------:|----------------:|---------------------:|--------------------------:|:---------:|------------------:|
| 0.1x | 147.6 | ✓ | 1.036 | **85** (over-V) | 0.038 | 28.1 | ✓ | 0.044 |
| 0.5x | 737.8 | ✓ | 1.033 | **44** (over-V) | 0.023 | 15.8 | ✓ | 0.043 |
| 1.0x | 1475.7 | ✓ | 1.010 | 1 | 0 | 0 | ✓ | 0.044 |
| 1.5x | 2213.5 | ✓ | 0.980 | 0 | 0.035 | 17.3 | ✓ | 0.048 |
| 2.0x | 2951.4 | ✓ | 0.926 | **31** (under-V) | 0.096 | 38.4 | ✓ | 0.059 |
| 3.0x | 4427.1 | **FAIL** | — | — | — | — | **FAIL** | — |

**PV-curve nose is between 2x and 3x (2951–4427 MW total load).** Both warm and
flat start fail at 3x and above. The AC PF equations have no solution past the nose;
this is a physical limit of the network, not a solver limitation. See
[`pf_helper.md` Section 9](../work-notes/pf_helper.md) for a detailed explanation.

**Two solution branches confirmed in PM.jl.** `warm_vs_flat` ≈ 0.044 pu for every
converged case, identical to the gap seen in GridKit (`pf_helper.ipynb` Section 14).
Warm-start always finds the high-voltage physical branch; flat-start always finds a
low-voltage branch ~0.044 pu lower. This is a structural property of the PF equations
for this network, not a GridKit artifact (see [`pf_helper.md` Section 7](../work-notes/pf_helper.md)).

**Voltage violations are a leading indicator, not simultaneous with collapse.**
- Under-loaded (0.1x, 0.5x): generators over-excite to hold PV setpoints with little
  reactive demand, pushing voltages above 1.05 pu on 44–85 buses.
- 1.5x: zero violations, extra reactive load pulls bus voltages down into the ANSI
  [0.95, 1.05] band, actually improving voltage profile vs base case.
- 2.0x: 31 buses below 0.95 pu. Voltage collapse precursors appear one scale step
  before the nose.

**The voltage-stiff (DC-approximation-valid, angle-dominated) regime breaks down well
before the nose.**
In `pf_helper.ipynb` Sections 10-13, within ±80% random per-bus load perturbation,
`max|dV| < 0.010 pu` and voltages barely moved. Here, with uniform 1.5x loading
(+50% total load), `max|dV| = 0.035 pu` and `max|dTheta| = 17.3 deg`, both voltage
and angle have moved substantially from base. The linearization assumed PV buses absorb
reactive changes without limit; past ~1.5x the generators begin hitting Q limits,
PV→PQ switching activates, and voltage magnitudes start moving alongside angles
(see [`pf_helper.md` Section 8](../work-notes/pf_helper.md) for the physical explanation).
By 2.0x, `max|dV| = 0.096 pu` and `max|dTheta| = 38 deg`: the response is strongly
nonlinear in both voltage and angle.


## Section 8b findings: gen outage sweep — voltage collapse precedes mathematical failure

All numbers come directly from `gen_off_df` above. Base case: 1475.7 MW total load,
total online generation ~1476 MW across ~49 generators (1 slack + 48 PV/PQ gens).

| N off | MW dropped | warm conv | warm V_min (pu) | violations | dV vs base (pu) | dTheta vs base (deg) | flat conv | warm vs flat (pu) |
|------:|----------:|:---------:|----------------:|-----------:|----------------:|---------------------:|:---------:|------------------:|
| 1  | 154.8  | yes | 1.010 | 1   | 0.005 | 7.6  | yes | 0.045 |
| 2  | 288.7  | yes | 1.006 | 1   | 0.011 | 9.6  | yes | 0.045 |
| 5  | 648.5  | yes | 0.989 | 0   | 0.039 | 21.0 | yes | 0.048 |
| 10 | 919.7  | yes | 0.841 | **85**  | 0.177 | 29.0 | yes | 0.084 |
| 15 | 1020.8 | yes | 0.821 | **96**  | 0.197 | 31.3 | yes | 0.098 |
| 20 | 1062.7 | yes | 0.809 | **99**  | 0.208 | 32.4 | yes | 0.110 |
| 25 | 1086.9 | yes | 0.736 | **177** | 0.279 | 36.3 | **no** | — |
| 30 | 1095.6 | **no** | — | — | — | — | **no** | — |

### Why does the PF still "solve" with 74% of generation offline?

This is the key question. The short answer: **the slack bus has no MW capacity limit
in an AC PF formulation.** PM.jl's `solve_pf` formulates a zero-objective NLP: find
voltages (V, θ) satisfying Kirchhoff's current laws at every bus. The slack bus (type 3)
is the reference node with fixed |V| and θ=0; its real and reactive output emerge as a
consequence of the solution, not a constraint. There is no `Pg ≤ Pgmax` for the slack.

So when the 25 largest generators go offline (dropping 1087 MW), the PF equations ask:
"can the remaining generators plus an unconstrained slack supply 1476 MW to the loads?"
Mathematically, yes. Physically, a real slack generator cannot ramp from its base-case
output to 1000+ MW. This experiment measures **network structural limits**, not dispatch
feasibility.

### What the results actually show: voltage collapse is the real story

"Converges" does not mean "acceptable." The warm V_min column is the honest indicator:

- **N=1–2**: V_min ≈ 1.006–1.010 pu, 1 voltage violation each. The system is largely
  intact; the slack and remaining generators re-dispatch smoothly.
- **N=5**: V_min = 0.989 pu, **zero violations** (reactive demand draws voltages into band).
  Angles already large (21 deg), comparable to the 2.0x load scaling case.
- **N=10** (920 MW dropped, 62% of base load): V_min = **0.841 pu**, 85 buses below 0.95 pu.
  The 10 offline generators no longer hold their PV voltage setpoints; those buses are now
  effectively PQ. This is a voltage emergency by any standard, but the PF equations still
  have a mathematical solution because the slack can supply the missing MW.
- **N=15–20**: V_min deteriorates to 0.809–0.821 pu, 96–99 violations. The warm-vs-flat
  gap grows from 0.044 to 0.110 pu: the two solution branches diverge as the system
  becomes more stressed, with the low-voltage branch dropping further each step.
- **N=25**: warm converges (V_min=0.736 pu, 177 violations), but flat-start **fails**.
  The low-voltage equilibrium no longer exists; only the high-voltage branch remains.
  With V_min at 0.74 pu and 88% of buses with violations, this solution is deeply
  unphysical but mathematically valid from the PF equations' perspective.
- **N=30**: no solution for either branch. The network equations have no equilibrium:
  the slack cannot route enough power to the loads given the remaining network topology
  and the reactive support lost from 30 generators going from PV to offline.

### Why the low-voltage branch disappears first (N=25 flat fails, warm still works)

In the load-scaling experiment (Section 8a), both branches failed simultaneously at 3x.
Here they fail asymmetrically. The flat-start (low-voltage) branch disappears at N=25
while the warm-start (high-voltage, physically operated) branch survives to N=30.
This suggests the two PV-curve noses are at different stress levels for this perturbation
type: the low-voltage branch is less robust to reactive support loss (which is what
generator outages primarily cause) than to uniform load increases.

### Two-branch gap grows with outage count

The warm-vs-flat gap of 0.044 pu is constant under moderate perturbations (Section 8a,
N=1 load-scale results, Section 6 all 14 perturbed cases). Here it grows monotonically:
0.045 (N=1) → 0.048 (N=5) → 0.084 (N=10) → 0.110 (N=20). Each generator outage removes
a voltage-supporting PV bus, shrinking the distance between the two equilibria until
the lower one merges with the upper and disappears.

### Voltage-stiff regime (angle-dominated): breaks earlier per MW than load scaling

| stress | max dV (pu) | max dTheta (deg) | regime |
|--------|------------|-----------------|--------|
| ±80% random load (Section 5) | 0.010 | 2.2 | voltage-stiff (DC-approx valid) |
| N=2 gen off (289 MW) | 0.011 | 9.6 | transitional (dTheta large) |
| N=5 gen off (649 MW) | 0.039 | 21.0 | nonlinear (voltage + angle both move) |
| 1.5x uniform load (Section 8a) | 0.035 | 17.3 | nonlinear |

Gen outages exit the voltage-stiff regime much earlier (per MW) than uniform
load scaling. A 289 MW gen outage produces dTheta=9.6 deg (Section 6 gen10rand_off:
306 MW produced only 16.5 deg in GridKit); the slack bus concentrates reactive support
loss at one location instead of distributing it across all buses.


# Section 9: cross-comparison of PM.jl vs GridKit on stress cases

Section 6 established that on the 14 mild perturbations (±80% random load, ±80%
wind curtailment, N≤10 random gens off), GridKit's voltage bias vs PM.jl was a
near-constant ~0.030 pu offset, so relative-delta QoIs were safe.

Section 8a/8b (PM.jl only) and `pf_helper.ipynb` Section 8 (GridKit only) then
pushed both solvers into stress cases: uniform load scaling (0.1x-8x) and largest-gen
outages (N=1-30). This section combines them.

**No re-solving here** — both solvers already wrote per-bus `*_solved.m` files to
`pm-solver/m-cases/stress-test/`:
- GridKit: `case_ACTIVSg200_load{s}x_solved.m` and `..._flat_solved.m` (from
  `pf_helper.ipynb` Section 8, via `run_solve_pf_out` / `run_solve_pf_out_flat`).
- PM.jl: `case_ACTIVSg200_load{s}x_pm_solved.m` and `..._pm_flat_solved.m` (from
  Section 8a/8b above, via `run_pm_solve_out`).

Section 9 just reads both sides with `parse_raw_m_bus` and computes per-bus
`|dV|`, `|dTheta|`.



In [27]:
from pf_utils import parse_raw_m_bus


def _diff_solved_m(label, pm_solved_m, gk_solved_m):
    """Compare PM.jl and GridKit solved .m files bus-by-bus. No solving."""
    if not pm_solved_m.exists():
        return {"case": label, "status": "missing_pm_solved"}
    if not gk_solved_m.exists():
        return {"case": label, "status": "missing_gk_solved"}
    pm_df = parse_raw_m_bus(pm_solved_m)
    gk_df = parse_raw_m_bus(gk_solved_m)
    cmp = pm_df.merge(
        gk_df[["bus_i", "V_pu", "theta_deg"]].rename(
            columns={"V_pu": "V_gk", "theta_deg": "theta_gk"}
        ),
        on="bus_i",
    )
    dV = (cmp.V_pu - cmp.V_gk).abs()
    dTheta = (cmp.theta_deg - cmp.theta_gk).abs()
    return {
        "case": label,
        "status": "ok",
        "pm_V_min": float(cmp.V_pu.min()),
        "gk_V_min": float(cmp.V_gk.min()),
        "pm_viols": int(((cmp.V_pu < 0.95) | (cmp.V_pu > 1.05)).sum()),
        "gk_viols": int(((cmp.V_gk < 0.95) | (cmp.V_gk > 1.05)).sum()),
        "max_dV_pu": float(dV.max()),
        "mean_dV_pu": float(dV.mean()),
        "max_dTheta_deg": float(dTheta.max()),
        "mean_dTheta_deg": float(dTheta.mean()),
    }


# --- 9a: load-scale sweep (warm-start on both sides) ---
pm_vs_gk_load_rows = []
for s in LOAD_SCALES:
    pm_solved = ST_DIR / f"case_ACTIVSg200_load{s}x_pm_solved.m"
    gk_solved = ST_DIR / f"case_ACTIVSg200_load{s}x_solved.m"
    row = _diff_solved_m(f"load{s}x", pm_solved, gk_solved)
    row["scale"] = s
    pm_vs_gk_load_rows.append(row)
    print(f"  [{s}x]  {row['status']}")

pm_vs_gk_load_df = pd.DataFrame(pm_vs_gk_load_rows)
pm_vs_gk_load_df

  [0.1x]  ok
  [0.5x]  ok
  [1.0x]  ok
  [1.5x]  ok
  [2.0x]  ok
  [3.0x]  missing_pm_solved
  [4.0x]  missing_pm_solved
  [6.0x]  missing_pm_solved
  [8.0x]  missing_pm_solved


,case,status,pm_V_min,gk_V_min,pm_viols,gk_viols,max_dV_pu,mean_dV_pu,max_dTheta_deg,mean_dTheta_deg,scale
0,load0.1x,ok,1.035678,1.028977,85.000000,0.000000,0.037315,0.008580,0.355939,0.234517,0.100000
1,load0.5x,ok,1.033409,1.028977,44.000000,0.000000,0.035464,0.007629,0.211803,0.089591,0.500000
2,load1.0x,ok,1.010229,1.009125,1.000000,0.000000,0.029728,0.005057,0.190703,0.073286,1.000000
3,load1.5x,ok,0.979793,0.980611,0.000000,0.000000,0.030475,0.004718,0.332523,0.141812,1.500000
4,load2.0x,ok,0.925815,0.928995,31.000000,21.000000,0.088971,0.009129,0.454267,0.147376,2.000000
5,load3.0x,missing_pm_solved,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.000000
6,load4.0x,missing_pm_solved,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.000000
7,load6.0x,missing_pm_solved,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6.000000
8,load8.0x,missing_pm_solved,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8.000000


In [28]:
# --- 9b: gen-outage sweep (warm-start on both sides) ---
pm_vs_gk_gen_rows = []
for n in GEN_OFF_NS:
    pm_solved = ST_DIR / f"case_ACTIVSg200_gen{n}largest_off_pm_solved.m"
    gk_solved = ST_DIR / f"case_ACTIVSg200_gen{n}largest_off_solved.m"
    row = _diff_solved_m(f"gen{n}largest_off", pm_solved, gk_solved)
    row["n_off"] = n
    pm_vs_gk_gen_rows.append(row)
    print(f"  [N={n:>2}]  {row['status']}")

pm_vs_gk_gen_df = pd.DataFrame(pm_vs_gk_gen_rows)
pm_vs_gk_gen_df

  [N= 1]  ok
  [N= 2]  ok
  [N= 5]  ok
  [N=10]  ok
  [N=15]  ok
  [N=20]  ok
  [N=25]  ok
  [N=30]  missing_pm_solved


,case,status,pm_V_min,gk_V_min,pm_viols,gk_viols,max_dV_pu,mean_dV_pu,max_dTheta_deg,mean_dTheta_deg,n_off
0,gen1largest_off,ok,1.009945,1.009002,1.000000,0.000000,0.029418,0.004928,0.195930,0.083642,1
1,gen2largest_off,ok,1.006312,1.007489,1.000000,0.000000,0.028799,0.004743,0.202037,0.097841,2
2,gen5largest_off,ok,0.989179,1.002334,0.000000,0.000000,0.038992,0.008632,0.172357,0.058225,5
3,gen10largest_off,ok,0.840948,0.995549,85.000000,0.000000,0.177366,0.063538,3.848515,1.616685,10
4,gen15largest_off,ok,0.820917,0.993792,96.000000,0.000000,0.196799,0.077324,4.855888,2.232930,15
5,gen20largest_off,ok,0.809003,0.993050,99.000000,0.000000,0.208330,0.088791,5.503377,2.627589,20
6,gen25largest_off,ok,0.736386,0.992528,177.000000,0.000000,0.278661,0.151850,9.724978,4.992483,25
7,gen30largest_off,missing_pm_solved,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,30


In [29]:
# --- 9c: summary against Section 6 baseline ---
# Base-case PM.jl vs GridKit max|dV| (from Section 5) = 0.030 pu
# Section 6 perturbed-case max|dV| range = [0.027, 0.030] pu (variation < 0.003 pu)
# The question here: does the offset stay constant in the stress regime?

BASE_OFFSET_PU = float(pm_vs_gk.dV.max())  # base-case PM vs GK max|dV|

print(f"Section 5 base case         max|dV| = {BASE_OFFSET_PU:.4f} pu")
print(
    f"Section 6 perturbed range   max|dV| in [{sweep_df.max_dV_pu.min():.4f}, {sweep_df.max_dV_pu.max():.4f}] pu"
)
print()

_load_ok = pm_vs_gk_load_df[pm_vs_gk_load_df.status == "ok"]
if not _load_ok.empty:
    print("Section 9a load-scale stress:")
    print(
        f"  max|dV|   range = [{_load_ok.max_dV_pu.min():.4f}, {_load_ok.max_dV_pu.max():.4f}] pu"
    )
    print(
        f"  max|dTheta| range = [{_load_ok.max_dTheta_deg.min():.4f}, {_load_ok.max_dTheta_deg.max():.4f}] deg"
    )
print()

_gen_ok = pm_vs_gk_gen_df[pm_vs_gk_gen_df.status == "ok"]
if not _gen_ok.empty:
    print("Section 9b gen-outage stress:")
    print(
        f"  max|dV|   range = [{_gen_ok.max_dV_pu.min():.4f}, {_gen_ok.max_dV_pu.max():.4f}] pu"
    )
    print(
        f"  max|dTheta| range = [{_gen_ok.max_dTheta_deg.min():.4f}, {_gen_ok.max_dTheta_deg.max():.4f}] deg"
    )

Section 5 base case         max|dV| = 0.0297 pu
Section 6 perturbed range   max|dV| in [0.0277, 0.0299] pu

Section 9a load-scale stress:
  max|dV|   range = [0.0297, 0.0890] pu
  max|dTheta| range = [0.1907, 0.4543] deg

Section 9b gen-outage stress:
  max|dV|   range = [0.0288, 0.2787] pu
  max|dTheta| range = [0.1724, 9.7250] deg


## Section 9 findings

All numbers below come directly from `pm_vs_gk_load_df` and `pm_vs_gk_gen_df` above,
and from Section 5/6 (base case + 14 perturbed cases).

### 9a load-scale stress

| scale | PM V_min | GK V_min | PM viols | GK viols | max\|dV\| (pu) | max\|dTheta\| (deg) |
|-----:|--------:|--------:|--------:|--------:|--------------:|--------------------:|
| 0.1x | 1.036 | 1.029 | **85** | 0 | 0.0373 | 0.36 |
| 0.5x | 1.033 | 1.029 | **44** | 0 | 0.0355 | 0.21 |
| 1.0x | 1.010 | 1.009 | 1 | 0 | 0.0297 | 0.19 |
| 1.5x | 0.980 | 0.981 | 0 | 0 | 0.0305 | 0.33 |
| 2.0x | 0.926 | 0.929 | **31** | **21** | 0.0890 | 0.45 |
| 3.0x - 8.0x | PM.jl did not converge (past PV-curve nose, see Section 8a) |

**The 0.030 pu constant offset holds only in a narrow band around 1.0x (0.5x-1.5x).**
Outside that band, on both sides:
- **Under-loaded (0.1x, 0.5x)**: PM.jl reports 44 and 85 over-voltage violations
  (V > 1.05 pu); GridKit reports zero. Under light load, physical generators must
  absorb reactive power to prevent buses from rising; when they hit Qmin they switch
  to PQ and voltage climbs above setpoint. GridKit's non-enforcement of Qmin lets its
  generators absorb unlimited reactive, so voltages stay clamped at the setpoint. The
  Q-limit gap is symmetric: it hides over-voltage at low load and under-voltage at
  high load.
- **2.0x**: max|dV| jumps from 0.030 to 0.089 pu (3x baseline), max|dTheta| from
  0.19 to 0.45 deg. GridKit still gets the qualitative picture (21 vs 31 violations,
  same V_min to within 0.003 pu), but the constant-offset assumption has broken.

### 9b gen-outage stress: sharp transition between N=5 and N=10

| N off | PM V_min | GK V_min | PM viols | GK viols | max\|dV\| (pu) | max\|dTheta\| (deg) |
|-----:|--------:|--------:|--------:|--------:|--------------:|--------------------:|
| 1  | 1.010 | 1.009 | 1 | 0 | 0.0294 | 0.20 |
| 2  | 1.006 | 1.007 | 1 | 0 | 0.0288 | 0.20 |
| 5  | 0.989 | 1.002 | 0 | 0 | 0.0390 | 0.17 |
| 10 | **0.841** | **0.996** | **85** | **0** | **0.177** | **3.85** |
| 15 | 0.821 | 0.994 | 96 | 0 | 0.197 | 4.86 |
| 20 | 0.809 | 0.993 | 99 | 0 | 0.208 | 5.50 |
| 25 | 0.736 | 0.993 | **177** | **0** | **0.279** | 9.72 |
| 30 | PM.jl did not converge (network has no equilibrium, see Section 8b) |

**The sharp transition between N=5 and N=10 is the operational boundary of
GridKit's usability on ACTIVSg200.**
- N=1, 2: baseline 0.029 pu offset (matches Section 5).
- N=5: 0.039 pu, first mild expansion.
- **N=10: 0.177 pu (6x baseline), max|dTheta| jumps from 0.17 deg to 3.85 deg.**
- N=25: 0.279 pu, max|dTheta| = 9.72 deg. GridKit reports V_min = 0.993 pu with
  zero violations while PM.jl reports V_min = 0.736 pu with 177 violations. The two
  solvers now report **qualitatively different network states**.

The mechanism is exactly the Q-limit non-enforcement documented in `pf_helper.md`
Section 1: dropping generators strands reactive support at neighboring PV buses;
those buses hit Qmax in PM.jl and switch to PQ (voltages sag); in GridKit they
keep holding their setpoints, so the network appears artificially stiff.

### Threshold-based verdict (matches Section 7 style)

| threshold | 9a violated at | 9b violated at |
|-----------|---------------|----------------|
| max\|dV\| < 0.010 pu (`DV_ABS_THRESH_PU`) | never (baseline offset already fails) | never (same reason) |
| max\|dV\| < 0.030 pu (base-case bias, ΔV cancellation limit) | already at 0.1x (viols mismatch); numerically at 2.0x | **N=5 borderline (0.039); breaks at N=10 (0.177)** |
| max\|dTheta\| < 1.0 deg (Section 7 angle threshold) | never | **breaks at N=10 (3.85 deg)** |

### Boundary of GridKit usability (revised)

The Section 7 verdict ("angles PASS, ΔV PASS, absolute V FAIL") is confirmed only
for the 14 mild perturbations (Section 6: ±80% random load, ±80% wind, N≤10 random
gens off). It **does not extend to**:
- Uniform load < 0.5x or > 1.5x (over/under-voltage violations mis-counted).
- Uniform load ≥ 2.0x (max|dV| grows past 0.030 pu).
- N ≥ 10 largest gens off (max|dV| and max|dTheta| both explode; violation counts
  qualitatively wrong).

### Implication for the 8760-scenario aleatoric UQ pipeline

Any hour where actual generator dispatch pushes several large units near Qmax
(peak-load nights, summer heat waves, low-wind + high-load coincidence) will land
in the GridKit-invalid regime. **PM.jl must be the production PF solver.** GridKit
`solve_pf` remains available for:
- Base-case sanity checks in the voltage-stiff regime.
- Angle-only QoIs (line flows) on cases within the Section 6 perturbation envelope.

This closes the top-priority open item from `pf_helper.md` Section 16.

